# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaniqbalwah/flyrank-ml-internship-arslan/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I am building a simple feature vector using the numerical features impressions_90d, sessions_90d, and content_age_days. Missing values for traffic metrics are filled with 0, and age is filled with the median. We also create our proxy label is_declining based on trend_direction.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Define features and handle missing values
features = ['impressions_90d', 'sessions_90d', 'content_age_days']
X = df[features].copy()
X['impressions_90d'] = X['impressions_90d'].fillna(0)
X['sessions_90d'] = X['sessions_90d'].fillna(0)
X['content_age_days'] = X['content_age_days'].fillna(X['content_age_days'].median())

# Define target label
y = df['trend_direction'] == 'down'

print("Feature Vector (X) shape:", X.shape)
print("Target Label (y) shape:", y.shape)
display(X.head())

Feature Vector (X) shape: (30000, 3)
Target Label (y) shape: (30000,)


,impressions_90d,sessions_90d,content_age_days
0,3803,17,187
1,15320,9,445
2,12581,11,141
3,11751,78,463
4,19140,145,263


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

impressions_90d: Total search appearances in the last 90 days. Missing = filled with 0. Exists before the prediction moment.

sessions_90d: Total clicks/visits in the last 90 days. Missing = filled with 0. Exists before the prediction moment.

content_age_days: Days since the page was published. Missing = filled with median. Exists before the prediction moment.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Checking for any remaining missing values in the feature vector:")
print(X.isnull().sum())
print("\nData types of features:")
print(X.dtypes)


Checking for any remaining missing values in the feature vector:
impressions_90d     0
sessions_90d        0
content_age_days    0
dtype: int64

Data types of features:
impressions_90d     int64
sessions_90d        int64
content_age_days    int64
dtype: object


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I am hunting for data leakage by checking the correlation between our features and the target label (is_declining). If any feature has a near 1.0 or -1.0 correlation, it's a massive red flag that the label has "leaked" into the feature. Our features should be predictive, but not perfect guarantees.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Combine X and y temporarily to check correlation
leakage_check_df = X.copy()
leakage_check_df['is_declining_label'] = y

print("Correlation with the target label (Leakage Hunt):")
correlations = leakage_check_df.corr()['is_declining_label'].drop('is_declining_label')
print(correlations.round(3))

if (correlations.abs() > 0.95).any():
    print("\nWARNING: Possible data leakage detected! High correlation.")
else:
    print("\nSAFE: No obvious data leakage detected in these features.")


Correlation with the target label (Leakage Hunt):
impressions_90d    -0.018
sessions_90d       -0.023
content_age_days   -0.164
Name: is_declining_label, dtype: float64

SAFE: No obvious data leakage detected in these features.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

client_id & content_id: Excluded because they are context identifiers. We want the model to learn universal decay patterns, not memorize specific clients.

trend_direction: Excluded from the feature vector because it is the exact column used to create our target label. Including it would cause 100% data leakage.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_columns = ['client_id', 'content_id', 'trend_direction']
print(f"Excluded columns: {excluded_columns}")

# Verify they are NOT in X
leak_test = any(col in X.columns for col in excluded_columns)
print(f"Are any excluded columns secretly in our feature vector X? {leak_test}")

Excluded columns: ['client_id', 'content_id', 'trend_direction']
Are any excluded columns secretly in our feature vector X? False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.